# AdiletCodex - QC & EDA
Quality control and exploratory analysis of the parsed current Codes of the Republic of Kazakhstan (trilingual: Kazakh / Russian / English), from adilet.zan.kz.

Checks: coverage, structural hierarchy, article-length distribution, cross-language alignment, and **language integrity** (that KZ/RU/EN are genuinely different languages, not duplicates).

In [1]:

import json, glob, os, re, unicodedata
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RAW = "../data/raw"
FIGS = "figs"; os.makedirs(FIGS, exist_ok=True)

ART_PAT = {"rus": r"статья\s+(\d+(?:-\d+)?)",
           "kaz": r"(\d+(?:-\d+)?)-бап",
           "eng": r"article\s+(\d+(?:-\d+)?)"}

def art_no(title, lang):
    m = re.match(r"^\s*" + ART_PAT.get(lang, r"(\d+)"), title or "", re.I)
    return m.group(1) if m else None

rows = []
for f in sorted(glob.glob(RAW + "/*.json")):
    base = os.path.basename(f)[:-5]
    parts = base.split("-")
    doc_id, lang = parts[0], parts[1]
    short = "-".join(parts[2:])
    try:
        d = json.load(open(f, encoding="utf-8"))
    except Exception:
        d = []
    for r in d:
        txt = r.get("article_text") or ""
        rows.append({
            "doc_id": doc_id, "short": short, "lang": lang,
            "part": r.get("part"), "section": r.get("section"), "chapter": r.get("chapter"),
            "article_no": art_no(r.get("article_title"), lang),
            "article_title": r.get("article_title"), "text": txt, "char_len": len(txt),
            "n_par": len(r.get("paragraphs") or []), "n_notes": len(r.get("notes") or []),
            "n_links": len(r.get("links") or []),
            "title_doc": r.get("title"), "status": r.get("status"), "url": r.get("url"),
        })
df = pd.DataFrame(rows)
print("Total article records:", len(df))
print("Distinct codes:", df.short.nunique(), "| languages:", sorted(df.lang.unique()))
df.head(3)


Total article records: 29568
Distinct codes: 24 | languages: ['eng', 'kaz', 'rus']


,doc_id,short,lang,part,section,chapter,article_no,article_title,text,char_len,n_par,n_notes,n_links,title_doc,status,url
0,K030000442_,land,eng,None,None,Chapter 1. General provisions,1,Article 1. Land fund of the Republic of Kazakh...,"In accordance with the designation, the land f...",939,2,2,0,Land Code of the Republic of Kazakhstan,Updated,https://old.adilet.zan.kz/eng/docs/K030000442_
1,K030000442_,land,eng,None,None,Chapter 1. General provisions,2,"Article 2. Rating of lands in categories, thei...","Rating of lands in categories, mentioned in Ar...",545,1,1,0,Land Code of the Republic of Kazakhstan,Updated,https://old.adilet.zan.kz/eng/docs/K030000442_
2,K030000442_,land,eng,None,None,Chapter 1. General provisions,3,Article 3. Land ownership,Land in the Republic of Kazakhstan shall be in...,443,1,1,0,Land Code of the Republic of Kazakhstan,Updated,https://old.adilet.zan.kz/eng/docs/K030000442_


## 1. Coverage: records per language and per code

In [2]:

print("Records per language:")
print(df.lang.value_counts().to_string())
piv = df.pivot_table(index="short", columns="lang", values="article_title",
                     aggfunc="count", fill_value=0)
piv = piv[["rus","kaz","eng"]] if set(["rus","kaz","eng"]).issubset(piv.columns) else piv
piv.loc["TOTAL"] = piv.sum()
piv


Records per language:
lang
eng    10330
rus     9623
kaz     9615


lang,rus,kaz,eng
short,,,
admin_offences,1077,1078,1078
admin_proc,195,195,195
budget,176,176,176
civil_general,425,424,1161
civil_proc,490,489,490
civil_special,744,741,743
construction,148,148,148
criminal,504,504,503
criminal_exec,178,178,178


## 2. Article length distribution (characters)

In [3]:

print(df.groupby("lang")["char_len"].describe()[["count","mean","50%","max"]])
fig, ax = plt.subplots(figsize=(7,4))
for lang in ["rus","kaz","eng"]:
    s = df[df.lang==lang]["char_len"].clip(upper=6000)
    ax.hist(s, bins=50, alpha=0.5, label=lang)
ax.set_xlabel("article length (chars, clipped at 6000)"); ax.set_ylabel("articles")
ax.legend(); ax.set_title("Article length distribution by language")
plt.tight_layout(); plt.savefig(FIGS+"/len_by_lang.png", dpi=110); plt.show()
print("empty-text records:", int((df.char_len==0).sum()))


        count         mean     50%       max
lang                                        
eng   10330.0  1900.060697  1097.5  121355.0
kaz    9615.0  1949.424857  1132.0  120588.0
rus    9623.0  1990.255326  1159.0  118004.0


empty-text records: 19


## 3. Structural hierarchy coverage
Share of articles that carry part / section / chapter context.

In [4]:

cov = df.assign(has_part=df.part.notna(), has_sec=df.section.notna(), has_ch=df.chapter.notna())
print((cov.groupby("lang")[["has_part","has_sec","has_ch"]].mean()*100).round(1).astype(str)+" %")


     has_part has_sec  has_ch
lang                         
eng    66.9 %  83.7 %  94.9 %
kaz    57.0 %  67.4 %  96.0 %
rus    64.3 %  84.5 %  97.0 %


## 4. Language integrity (the key check)
KZ must contain Kazakh-specific letters; RU Cyrillic without them; EN Latin. And no two languages of the same article may be byte-identical (that was the failure in the earlier Uzbek corpus).

In [5]:

KAZ = set("әғқңөұүһіӘҒҚҢӨҰҮҺІ")
def profile(t):
    t = (t or "")[:800]
    cyr = sum(1 for c in t if "CYRILLIC" in unicodedata.name(c, ""))
    lat = sum(1 for c in t if "LATIN" in unicodedata.name(c, ""))
    kaz = any(c in KAZ for c in t)
    return cyr, lat, kaz
prof = df.assign(**{k: v for k, v in zip(["cyr","lat","kaz_letters"],
        zip(*df.text.map(profile)))})
# expected script per lang
def ok(row):
    if row.lang=="eng": return row.lat > row.cyr
    if row.lang=="kaz": return row.cyr > row.lat  # kaz is Cyrillic script
    if row.lang=="rus": return row.cyr > row.lat
    return True
prof["script_ok"] = prof.apply(ok, axis=1)
print("script matches expected, per lang (%):")
print((prof.groupby("lang")["script_ok"].mean()*100).round(2).to_string())
print("\nKZ records actually containing Kazakh-specific letters: %.1f%%" %
      (prof[prof.lang=="kaz"]["kaz_letters"].mean()*100))
# cross-language identical-text check
key = ["short","article_no"]
wide = df[df.article_no.notna()].pivot_table(index=key, columns="lang", values="text",
        aggfunc="first")
def same(a,b): return (a is not None) and (b is not None) and isinstance(a,str) and a==b
dups = 0; checked = 0
for _, r in wide.iterrows():
    for a,b in [("rus","kaz"),("rus","eng"),("kaz","eng")]:
        if a in wide.columns and b in wide.columns:
            checked += 1
            if same(r.get(a), r.get(b)): dups += 1
print("\ncross-language identical-text pairs:", dups, "(out of", checked, "compared) -> want 0")


script matches expected, per lang (%):
lang
eng    99.88
kaz    99.98
rus    99.95

KZ records actually containing Kazakh-specific letters: 100.0%



cross-language identical-text pairs: 9 (out of 31092 compared) -> want 0


## 5. Cross-language alignment
How many articles of each code are present in all three languages (matched by article number).

In [6]:

align = (df[df.article_no.notna()]
         .groupby(["short","lang"])["article_no"].apply(set).unstack())
def tri(row):
    sets = [row[l] for l in ["rus","kaz","eng"] if l in row.index and isinstance(row[l], set)]
    if len(sets) < 3: return np.nan
    inter = set.intersection(*sets); uni = set.union(*sets)
    return round(100*len(inter)/len(uni), 1) if uni else np.nan
rep = pd.DataFrame({
    "rus": align.get("rus").map(lambda s: len(s) if isinstance(s,set) else 0),
    "kaz": align.get("kaz").map(lambda s: len(s) if isinstance(s,set) else 0),
    "eng": align.get("eng").map(lambda s: len(s) if isinstance(s,set) else 0),
})
rep["3-lang overlap %"] = align.apply(tri, axis=1)
rep.sort_values("3-lang overlap %")


,rus,kaz,eng,3-lang overlap %
short,,,,
civil_general,425,424,1159,36.6
subsoil,301,301,292,97.0
labor,227,227,223,97.4
ecological,426,426,418,98.1
social,273,270,272,98.5
health,301,300,298,98.7
customs,590,590,583,98.8
civil_special,744,741,743,98.9
admin_offences,1077,1078,1076,99.1


## 6. Content sanity: duplicates, gaps, empties

In [7]:

dup = df[df.article_no.notna()].groupby(["short","lang","article_no"]).size()
dupd = dup[dup>1]
print("duplicate (code,lang,article_no) keys:", len(dupd))
print(dupd.head(10).to_string() if len(dupd) else "  none")
print("\nempty article_text:", int((df.char_len==0).sum()))
# numbering gaps per code (rus), base numbers only
def gaps(short):
    s = df[(df.short==short)&(df.lang=="rus")]["article_no"].dropna()
    base = sorted({int(x.split('-')[0]) for x in s})
    if not base: return 0
    full = set(range(base[0], base[-1]+1))
    return len(full - set(base))
gp = pd.Series({sh: gaps(sh) for sh in sorted(df.short.unique())}, name="rus_numbering_gaps")
print("\nnumbering gaps (repealed articles) per code - expected, not errors:")
print(gp.to_string())


duplicate (code,lang,article_no) keys: 14
short           lang  article_no
admin_offences  eng   187           2
                      404           2
civil_general   eng   1123          2
                      86            2
criminal        kaz   51            2
labor           eng   143           2
subsoil         eng   120           2
                      126           2
                      143           2
                      213           2

empty article_text: 19

numbering gaps (repealed articles) per code - expected, not errors:
admin_offences      2
admin_proc          0
budget              0
civil_general       0
civil_proc         39
civil_special       0
construction        0
criminal            0
criminal_exec       3
criminal_proc      22
customs             0
digital             0
ecological          0
entrepreneurial    24
forest              0
health              0
labor               0
land                0
marriage_family    25
social              0
subsoil     

## 7. Random samples (eyeball check)

In [8]:

import random
random.seed(2026)
for _ in range(4):
    r = df.iloc[random.randrange(len(df))]
    print("="*80)
    print(f"[{r.lang}] {r.short}  |  {r.article_title}")
    print(f"part={r.part} | section={r.section} | chapter={r.chapter}")
    print(r.text[:400])
    print()


[eng] criminal_proc  |  Article 585. Detention during the transit and temporary extradition of a person (extradition)
part=Special Part | section=Section 12. International cooperation in criminal proceedings | chapter=Chapter 60. Extradition of persons (extradition)
The decision of the competent authority of a foreign state to detain a person in custody or his (her) sentence of imprisonment shall be grounds for detention in the territory of the Republic of Kazakhstan of the persons who:
1) transported in transit through the territory of the Republic of Kazakhstan;
2) temporarily extradited (extradited) to the Republic of Kazakhstan.


[eng] civil_proc  |  Article 176. The form and content of the civil agreement
part=None | section=SECTION 2. SUIT PROCEEDING | chapter=Chapter 17. SETTLEMENT ARRANGEMENTS
The settlement agreement shall be concluded in written form and signed by the parties or their representatives if they have the authority to enter into the settlement agreement specifica

## 8. Articles per code (bar chart)

In [9]:

by_code = df[df.lang=="rus"].groupby("short").size().sort_values()
fig, ax = plt.subplots(figsize=(8,7))
by_code.plot.barh(ax=ax)
ax.set_xlabel("articles (RU)"); ax.set_title("Articles per code (Russian)")
plt.tight_layout(); plt.savefig(FIGS+"/articles_per_code.png", dpi=110); plt.show()
print("saved figs/len_by_lang.png, figs/articles_per_code.png")


saved figs/len_by_lang.png, figs/articles_per_code.png
